# Stage 1: Dataset Understanding
Goal: Understand the dataset structure, features, target classes, and data quality before preprocessing.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/dataset.csv", low_memory=False)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (54783, 38)

Columns:
['timestamp_c', 'frame.number', 'frame.len', 'frame.protocols', 'wlan.duration', 'wlan.ra', 'wlan.ta', 'wlan.da', 'wlan.sa', 'wlan.bssid', 'wlan.frag', 'wlan.seq', 'llc.type', 'ip.hdr_len', 'ip.len', 'ip.id', 'ip.flags', 'ip.ttl', 'ip.proto', 'ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'tcp.seq_raw', 'tcp.ack_raw', 'tcp.hdr_len', 'tcp.flags', 'tcp.window_size', 'tcp.options', 'udp.srcport', 'udp.dstport', 'udp.length', 'data.data', 'data.len', 'wlan.fc.type', 'wlan.fc.subtype', 'time_since_last_packet', 'class']


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54783 entries, 0 to 54782
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   timestamp_c             54783 non-null  str  
 1   frame.number            54783 non-null  str  
 2   frame.len               54783 non-null  str  
 3   frame.protocols         54783 non-null  str  
 4   wlan.duration           54783 non-null  str  
 5   wlan.ra                 54783 non-null  str  
 6   wlan.ta                 54783 non-null  str  
 7   wlan.da                 54783 non-null  str  
 8   wlan.sa                 54783 non-null  str  
 9   wlan.bssid              54783 non-null  str  
 10  wlan.frag               54783 non-null  str  
 11  wlan.seq                54783 non-null  str  
 12  llc.type                54783 non-null  str  
 13  ip.hdr_len              54783 non-null  str  
 14  ip.len                  54783 non-null  str  
 15  ip.id                   54783 

In [4]:
df["class"].value_counts(dropna=False)

class
NaN           21679
Replay        12006
DoS attack    11671
benign         9425
class             2
Name: count, dtype: int64

# Initial Dataset Observations

### Dataset Overview
- Total number of samples (rows): **54,783**
- Total number of features (columns): **38**

### Data Type Issues
- All 38 columns have been loaded as `object` (string) data type.
- Several features such as `timestamp_c`, `frame.len`, `ip.ttl`, `tcp.srcport`, etc., are expected to be numeric.
- Appropriate data type conversion will be required during the preprocessing stage.

### Missing Values
- Missing values are present in multiple columns, particularly:
  - `ip.ttl`
  - `ip.proto`
  - `ip.src`
  - `ip.dst`
  - `time_since_last_packet`
  - `class`

### Target Variable (`class`) Distribution
| Class Label | Count |
|-------------|-------|
| Replay | 12,006 |
| DoS attack | 11,671 |
| benign | 9,425 |
| NaN | 21,679 |
| class | 2 |

### Potential Data Quality Issues
- The presence of `'class'` as a label is abnormal and suggests:
  - Corrupted rows in the dataset.
  - Repeated header rows within the CSV file.
  - Improper merging of multiple CSV files.

### Key Observation
- Approximately **39.6%** of the samples have missing class labels.
- The source and significance of these unlabeled samples must be investigated before performing any preprocessing, feature engineering, or model development.

### Questions for Further Investigation
1. Why are all columns stored as `object` data types?
2. Why do nearly 40% of the samples have missing labels?
3. Are the unlabeled samples associated with specific attack categories?
4. Was the dataset created by incorrectly merging multiple CSV files?
5. Can the missing labels be recovered?

In [5]:
df[df["class"] == "class"]

,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
13716,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
26362,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class


In [6]:
df[df["class"].isna()].head()

,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
9425,timestamp_p,height,x_speed,y_speed,z_speed,pitch,roll,yaw,temperature,distance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9426,28123.02875,80,0,0,0,0,0,0,68,84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9427,28123.55962,80,0,0,0,0,0,0,68,84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9428,28124.0769,80,0,0,0,0,0,0,68,83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9429,28124.602,80,0,0,0,-1,0,0,67,83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df[df["class"].isna()].tail()

,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
54778,12,61,45,104,0,24,179,0,0,0,...,-44.64154535,9.200630612,6.027533453,FDI,NaN,NaN,NaN,NaN,NaN,NaN
54779,12,61,45,104,0,24,179,0,0,0,...,-45.73679353,7.40427046,5.141048091,FDI,NaN,NaN,NaN,NaN,NaN,NaN
54780,12,61,45,104,0,24,179,0,0,0,...,-46.45840489,6.522716201,4.755015641,FDI,NaN,NaN,NaN,NaN,NaN,NaN
54781,12,61,46,106,0,24,180,0,0,0,...,-45.90483529,6.201081795,5.654590795,FDI,NaN,NaN,NaN,NaN,NaN,NaN
54782,12,62,51,106,1,25,178,1,0,0,...,-41.97397398,7.184936734,10.00216155,FDI,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.iloc[9400:9450]

,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
9400,54378.07704,802,26,0,60,1,0,1,0,0,...,0,0,0,0,0,0,2,12,0.042087,benign
9401,54378.13699,804,230,2,44,0,1,0,1,0,...,0,8889,8890,176,45,168,2,8,0.059947,benign
9402,54378.17944,806,26,0,60,1,0,1,0,0,...,0,0,0,0,0,0,2,12,0.042451,benign
9403,54381.55577,827,230,2,44,0,1,0,1,0,...,0,8889,8890,176,48,168,2,8,3.376332,benign
9404,54381.55862,829,26,0,60,1,0,1,0,0,...,0,0,0,0,0,0,2,12,0.002845,benign
9405,54381.65781,831,230,2,44,0,1,0,1,0,...,0,8889,8890,176,51,168,2,8,0.099196,benign
9406,54381.66106,833,26,0,60,1,0,1,0,0,...,0,0,0,0,0,0,2,12,0.003245,benign
9407,54385.07465,854,238,2,44,0,1,0,1,0,...,0,8889,8890,184,8,176,2,8,3.413589,benign
9408,54385.14254,856,26,0,60,1,0,1,0,0,...,0,0,0,0,0,0,2,12,0.067896,benign
9409,54385.17666,858,239,2,44,0,1,0,1,0,...,0,8889,8890,185,7,177,2,8,0.034114,benign


In [9]:
df.iloc[33090:33120]

,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,...,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
33090,28135.26789,952,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.030203,Replay
33091,28135.30087,953,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.032982,Replay
33092,28135.3251,954,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.024231,Replay
33093,28135.34671,955,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.021605,Replay
33094,28135.37874,956,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.032036,Replay
33095,28135.64491,957,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.26617,Replay
33096,28135.64739,958,26,0,314,1,0,1,0,0,...,0,0,0,0,0,0,0,12,0.002477,Replay
33097,28135.65055,959,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.003155,Replay
33098,28135.65275,960,26,0,314,1,0,1,0,0,...,0,0,0,0,0,0,0,12,0.0022,Replay
33099,28135.65612,961,26,0,314,0,1,0,1,0,...,0,0,0,0,0,0,0,12,0.003377,Replay


In [10]:
df[df["class"].isna()].index.min(), df[df["class"].isna()].index.max()

(np.int64(9425), np.int64(54782))

# Decision to Create a New Network Dataset

## Motivation

The original IEEE T-ITS dataset cannot be directly used for machine learning model development due to the following reasons:

1. **Mixed Data Modalities**
   - The dataset contains both:
     - Network communication features (`timestamp_c`, `wlan.*`, `ip.*`, `tcp.*`, `udp.*`)
     - UAV telemetry features (`timestamp_p`, `height`, `x_speed`, `pitch`, `yaw`, etc.)

2. **Inconsistent Schema**
   - Different sections of the CSV file follow different feature schemas.
   - Some rows represent network traffic, while others represent physical telemetry measurements.

3. **Data Quality Issues**
   - Presence of missing labels (`NaN`) in the `class` column.
   - Presence of repeated header rows (`class` appearing as a label).
   - All columns are loaded as `object` datatype due to schema inconsistencies.

4. **Project Scope**
   - The objective of this project is:

     **"AI-Based Intrusion Detection for UAV Communication Networks Using Machine Learning and Deep Learning"**

   - Therefore, the primary focus is on **network communication data** rather than physical telemetry data.

## Decision

A new dataset, namely `network_dataset.csv`, will be created by:

- Extracting only the network traffic records.
- Retaining all network-related features and the target label.
- Excluding telemetry records that do not align with the project's objective.

The original dataset (`dataset.csv`) will be preserved as the raw source dataset and will not be modified.

## Future Work

The excluded telemetry data may be utilized in future work to develop a **multimodal cyber-physical intrusion detection system** by combining network and physical UAV measurements.